# 00. Data, coverage, and validation

**Purpose.** Build the quarterly panel, show what is and is not covered, and
flag any figure that breaks comparability before a single chart is drawn.

**Why this notebook exists.** Eternal restated its reporting when Blinkit moved
to an inventory-led model. Reported revenue jumped with no matching change in
Net Order Value. A parser that pulls `revenue` across that break produces growth
charts that are wrong but entirely plausible-looking. Everything downstream
depends on catching it here.

**Read this first if you are reviewing the repo.** It establishes what the data
can and cannot support.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src import config, panel, metrics, viz

viz.apply_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)


## Load the panel

The panel is committed to the repo so every notebook is reproducible without
re-downloading filings. Source URL is mandatory on every row.


In [2]:
df = panel.load_panel()
print(f"{len(df)} quarters: {df['quarter'].iloc[0]} to {df['quarter'].iloc[-1]}")
df.head()

15 quarters: Q3FY23 to Q1FY27


,quarter,quarter_end,fd_gov,fd_nov,fd_adj_revenue,fd_contribution,fd_adj_ebitda,fd_mtu,fd_restaurant_partners_k,fd_delivery_partners_k,qc_gov,qc_nov,qc_revenue,qc_contribution,qc_adj_ebitda,qc_orders_mn,qc_aov,qc_naov,qc_mtu,qc_stores,go_nov,hp_revenue,others_adj_revenue,others_adj_ebitda,consol_adj_revenue,consol_adj_ebitda,bistro_kitchens,bistro_cities,source_url,_ord
0,Q3FY23,2022-12-31,6680.0,NaN,1565,339,23,17.4,209,330,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,421,NaN,NaN,NaN,-265,NaN,NaN,https://www.bseindia.com/xml-data/corpfiling/A...,94
1,Q4FY23,2023-03-31,6569.0,NaN,1530,380,78,16.6,215,316,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,478,NaN,NaN,NaN,-175,NaN,NaN,https://www.bseindia.com/xml-data/corpfiling/A...,95
2,Q1FY24,2023-06-30,7318.0,NaN,1742,466,181,17.5,226,352,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,617,NaN,NaN,NaN,12,NaN,NaN,https://www.bseindia.com/xml-data/corpfiling/A...,96
3,Q2FY24,2023-09-30,7980.0,NaN,1925,525,204,18.4,238,410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,745,NaN,NaN,NaN,41,NaN,NaN,https://www.bseindia.com/xml-data/corpfiling/A...,97
4,Q3FY24,2023-12-31,8486.0,NaN,2025,601,252,18.8,254,419,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,859,NaN,NaN,NaN,125,NaN,NaN,https://www.bseindia.com/xml-data/corpfiling/A...,98


## Coverage

Which fields are actually populated. Fill gaps manually from the filings before
proceeding: a silent NaN becomes a dropped row in a later merge and quietly
shortens the series.


In [3]:
cov = panel.coverage_report(df)
cov

,field,present,missing,coverage,status
20,bistro_kitchens,2,13,0.13,GAPS
6,qc_gov,6,9,0.40,GAPS
18,consol_adj_revenue,10,5,0.67,GAPS
17,others_adj_ebitda,10,5,0.67,GAPS
16,others_adj_revenue,10,5,0.67,GAPS
15,go_nov,10,5,0.67,GAPS
14,qc_stores,10,5,0.67,GAPS
13,qc_mtu,10,5,0.67,GAPS
12,qc_naov,10,5,0.67,GAPS
11,qc_orders_mn,10,5,0.67,GAPS


In [4]:
gaps = cov[cov["status"] != "ok"]
if len(gaps):
    print(f"{len(gaps)} field(s) incomplete. Resolve before analysis:")
    display(gaps)
else:
    print("Full coverage across all core fields.")

16 field(s) incomplete. Resolve before analysis:


,field,present,missing,coverage,status
20,bistro_kitchens,2,13,0.13,GAPS
6,qc_gov,6,9,0.40,GAPS
18,consol_adj_revenue,10,5,0.67,GAPS
17,others_adj_ebitda,10,5,0.67,GAPS
16,others_adj_revenue,10,5,0.67,GAPS
15,go_nov,10,5,0.67,GAPS
14,qc_stores,10,5,0.67,GAPS
13,qc_mtu,10,5,0.67,GAPS
12,qc_naov,10,5,0.67,GAPS
11,qc_orders_mn,10,5,0.67,GAPS


## Validation

Flag any quarter-on-quarter move outside a plausible band. Breaches are written
to `data/processed/review_queue.csv` rather than raised, because a definitional
change is a legitimate breach that needs a written explanation, not a crash.

Every row must be inspected, marked `reviewed=True`, and explained in
`ASSUMPTIONS.md` before the analysis notebooks will run.


In [5]:
breaches = panel.validate(df)
if breaches.empty:
    print("No validation breaches.")
else:
    print(f"{len(breaches)} breach(es) queued for review:")
    display(breaches)

1 breach(es) queued for review:


,quarter,field,prev_value,value,qoq_change,band,reviewed,explanation
0,Q4FY25,others_adj_ebitda,1.0,-16.0,-17.0,"[-3.0, 3.0]",True,Legitimate sign flip on a tiny base. Others ad...


### Expected breaches

The inventory-model transition should appear here as a large positive jump in
`consol_revenue` and `qc_revenue` with no corresponding move in `qc_nov`. That
is the signature of an accounting change rather than a business change.

If it does *not* appear, the parser is probably reading a restated series and
the break has been smoothed away. Check the filing footnotes directly.


## Derived metrics

What is derivable differs by segment, and the difference is a finding in itself.

**Quick commerce** discloses Orders and net average order value, so orders per
customer per month is available and NOV splits three ways into customers,
frequency and ticket size.

**Food delivery** has never disclosed Orders or AOV in the shareholders'
letters. Only NOV, GOV and monthly transacting customers. So frequency cannot be
separated from ticket size, and the honest substitute is NOV per transacting
customer, which carries both. The original plan to solve frequency from
`NOV = MTU x frequency x AOV x 3` is not executable for food delivery, and
saying so is better than inventing an AOV.

In [6]:
df = metrics.add_derived(df)
cols = [c for c in df.columns if c.endswith(("_per_mtu", "_frequency", "_pct_nov",
                                             "_take_rate", "_share", "_derived"))]
print("derived:", ", ".join(cols))
df[["quarter", *cols]].round(4).tail(6)

derived: qc_frequency, qc_naov_derived, fd_nov_per_mtu, fd_gov_per_mtu, fd_cm_pct_nov, qc_cm_pct_nov, fd_take_rate, qc_take_rate, qc_nov_share, fd_nov_share


,quarter,qc_frequency,qc_naov_derived,fd_nov_per_mtu,fd_gov_per_mtu,fd_cm_pct_nov,qc_cm_pct_nov,fd_take_rate,qc_take_rate,qc_nov_share,fd_nov_share
9,Q4FY25,3.4477,519.5483,3928.2297,4678.4689,0.1026,0.0393,0.2934,0.2321,0.4221,0.4708
10,Q1FY26,3.4852,520.8263,3915.7205,4702.6201,0.0987,0.0391,0.2963,0.2608,0.4560,0.4443
11,Q2FY26,3.5689,524.4275,3909.9585,NaN,0.1036,0.0464,0.3038,0.8469,0.5042,0.4068
12,Q3FY26,3.4364,546.6502,3954.2169,NaN,0.1039,0.0553,0.3101,0.9215,0.5168,0.3826
13,Q4FY26,3.3566,525.2282,3841.3386,NaN,0.1023,0.0544,0.3203,0.9198,0.5352,0.3630
14,Q1FY27,3.4696,517.5831,3959.1912,NaN,0.1019,0.0529,0.3284,0.9143,0.5505,0.3461


### Sanity check the derived series

Food delivery Orders and AOV are not disclosed, so order frequency cannot be
computed. Quick commerce discloses both, which gives a free integrity test: NOV
divided by Orders must reproduce the published net AOV.

## Save the enriched panel

Downstream notebooks read this file rather than recomputing, so every analysis
runs against an identical input.


In [7]:
config.ensure_dirs()
out = config.PROCESSED / "eternal_quarterly_enriched.csv"
df.drop(columns=["_ord"], errors="ignore").to_csv(out, index=False)
print(f"Saved to {out.relative_to(config.ROOT)}")

Saved to data/processed/eternal_quarterly_enriched.csv


---

**Next:** `01_unit_economics.ipynb` computes the switching value. It is the
hypothesis every other notebook now serves, so run it before the rest.